# Preparación y consolidación inicial de los datos ATP

Este notebook constituye la primera etapa del pipeline computacional del TFG. Su objetivo es reunir y preparar los archivos anuales de partidos del circuito ATP utilizados posteriormente en la construcción de la base de datos del estudio.

Se cargan los registros originales de partidos procedentes del repositorio `tennis_atp`, se integran las temporadas consideradas en un único conjunto de datos y se incorporan variables auxiliares que identifican la función temporal de cada observación dentro del estudio. La estructura temporal utilizada es:

- **2008 y 2021:** años auxiliares (*buffer*) empleados posteriormente para reconstruir el historial competitivo previo;
- **2009–2019 y 2022–2023:** temporadas destinadas al desarrollo del análisis y de los modelos;
- **2020:** temporada excluida del estudio;
- **2024:** temporada reservada para la evaluación temporal externa.

Por compatibilidad con el resto del pipeline, la variable interna `es_validacion_temporal` identifica las observaciones correspondientes a 2024, aunque metodológicamente esta temporada constituye la muestra de evaluación temporal externa y no forma parte de la validación interna de los modelos.

Finalmente, las variables originales se renombran mediante una nomenclatura homogénea y el conjunto consolidado se exporta como `atp_matches_2008_2019_2021_2024_full.csv`, que constituye la entrada de la siguiente etapa de preparación y transformación de los datos.

## 1. Configuración y periodo temporal

Se definen las rutas de trabajo y las temporadas empleadas en cada función metodológica del estudio. Los años 2008 y 2021 se conservan únicamente como *buffer*, mientras que 2024 queda identificado desde esta etapa como muestra de evaluación temporal externa.

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
# Carpeta donde están los CSV originales
DATA_DIR = Path("./tennis_atp-master")

# Carpeta donde guardaremos el dataset concatenado
INTERIM_DIR = Path("./interim")
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

print("Carpeta de datos:", DATA_DIR.resolve())
print("Carpeta de salida:", INTERIM_DIR.resolve())

Carpeta de datos: C:\Users\Usuario\Desktop\TFG\Datos\tennis_atp-master\tennis_atp-master
Carpeta de salida: C:\Users\Usuario\Desktop\TFG\Datos\tennis_atp-master\interim


In [3]:
ANIOS_BUFFER_CARGA = [
    2008,
    2021
]

ANIOS_DESARROLLO_MODELO = (
    list(range(2009, 2020))
    + [2022, 2023]
)

ANIO_VALIDACION_TEMPORAL = 2024

ANIOS_ESTUDIO = (
    ANIOS_DESARROLLO_MODELO
    + [ANIO_VALIDACION_TEMPORAL]
)

years = sorted(
    ANIOS_BUFFER_CARGA
    + ANIOS_ESTUDIO
)

print("Años cargados:")
print(years)

print("\nAños buffer para cargas:")
print(ANIOS_BUFFER_CARGA)

print("\nAños de desarrollo del modelo:")
print(ANIOS_DESARROLLO_MODELO)

print("\nAño de evaluación temporal externa:")
print(ANIO_VALIDACION_TEMPORAL)

print("\nAños de estudio:")
print(ANIOS_ESTUDIO)

Años cargados:
[2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2021, 2022, 2023, 2024]

Años buffer para cargas:
[2008, 2021]

Años de desarrollo del modelo:
[2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2022, 2023]

Año de evaluación temporal externa:
2024

Años de estudio:
[2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2022, 2023, 2024]


## 2. Carga y consolidación de los archivos anuales

Se cargan los archivos anuales de partidos ATP correspondientes al periodo operativo y se comprueba que todos estén disponibles y presenten el mismo esquema de variables antes de concatenarlos.

In [4]:
match_dfs = []
missing_files = []
columnas_referencia = None

for year in years:
    file_path = DATA_DIR / f"atp_matches_{year}.csv"

    if file_path.exists():
        df_year = pd.read_csv(
            file_path,
            low_memory=False
        )

        if columnas_referencia is None:
            columnas_referencia = df_year.columns.tolist()
        else:
            assert df_year.columns.tolist() == columnas_referencia, (
                f"El archivo de {year} no presenta el mismo esquema "
                "de columnas que el resto de temporadas."
            )

        df_year["match_year"] = year
        match_dfs.append(df_year)

        print(
            f"{year}: {df_year.shape[0]} filas, "
            f"{df_year.shape[1]} columnas"
        )

    else:
        missing_files.append(file_path.name)

print(
    "\nArchivos que faltan:",
    missing_files if missing_files else "ninguno"
)

assert not missing_files, (
    "Faltan archivos anuales. No continúes hasta "
    "tener también 2008 y 2021."
)

print("\nEsquema de columnas homogéneo en todas las temporadas.")

2008: 3123 filas, 50 columnas
2009: 3085 filas, 50 columnas
2010: 3030 filas, 50 columnas
2011: 3015 filas, 50 columnas
2012: 3009 filas, 50 columnas
2013: 2944 filas, 50 columnas
2014: 2901 filas, 50 columnas
2015: 2943 filas, 50 columnas
2016: 2941 filas, 50 columnas
2017: 2911 filas, 50 columnas
2018: 2897 filas, 50 columnas
2019: 2806 filas, 50 columnas
2021: 2733 filas, 50 columnas
2022: 2917 filas, 50 columnas
2023: 2986 filas, 50 columnas
2024: 3076 filas, 50 columnas

Archivos que faltan: ninguno

Esquema de columnas homogéneo en todas las temporadas.


## 3. Clasificación temporal y homogeneización

Los registros anuales se integran en un único conjunto y se incorporan indicadores que identifican la función de cada temporada dentro del estudio. Posteriormente, las variables originales se renombran mediante una nomenclatura homogénea utilizada en las siguientes etapas del pipeline.

In [5]:
matches_all = pd.concat(
    match_dfs,
    ignore_index=True,
    sort=False
)

matches_all["es_muestra_estudio"] = (
    matches_all["match_year"]
    .isin(ANIOS_ESTUDIO)
    .astype(int)
)

matches_all["es_buffer_carga"] = (
    matches_all["match_year"]
    .isin(ANIOS_BUFFER_CARGA)
    .astype(int)
)
matches_all["es_desarrollo_modelo"] = (
    matches_all["match_year"]
    .isin(ANIOS_DESARROLLO_MODELO)
    .astype(int)
)

matches_all["es_validacion_temporal"] = (
    matches_all["match_year"]
    .eq(ANIO_VALIDACION_TEMPORAL)
    .astype(int)
)

assert (
    matches_all["es_muestra_estudio"]
    + matches_all["es_buffer_carga"]
).eq(1).all()

assert (
    matches_all["es_desarrollo_modelo"]
    + matches_all["es_validacion_temporal"]
).eq(matches_all["es_muestra_estudio"]).all()

assert (
    matches_all["es_buffer_carga"]
    + matches_all["es_desarrollo_modelo"]
    + matches_all["es_validacion_temporal"]
).eq(1).all()

assert matches_all.loc[
    matches_all["match_year"].isin(ANIOS_BUFFER_CARGA),
    "es_buffer_carga"
].eq(1).all()

assert matches_all.loc[
    matches_all["match_year"].eq(ANIO_VALIDACION_TEMPORAL),
    "es_validacion_temporal"
].eq(1).all()

print("\nDistribución metodológica:")

display(
    matches_all.groupby(
        [
            "match_year",
            "es_buffer_carga",
            "es_desarrollo_modelo",
            "es_validacion_temporal",
            "es_muestra_estudio"
        ]
    )
    .size()
    .rename("numero_partidos")
    .to_frame()
)


Distribución metodológica:


,,,,,numero_partidos
match_year,es_buffer_carga,es_desarrollo_modelo,es_validacion_temporal,es_muestra_estudio,
2008,1,0,0,0,3123
2009,0,1,0,1,3085
2010,0,1,0,1,3030
2011,0,1,0,1,3015
2012,0,1,0,1,3009
2013,0,1,0,1,2944
2014,0,1,0,1,2901
2015,0,1,0,1,2943
2016,0,1,0,1,2941


## 4. Renombrado de variables

Las variables originales del conjunto ATP se renombran mediante una nomenclatura homogénea y más descriptiva, manteniendo intacta su información. Este cambio facilita la interpretación de los datos y permite utilizar nombres consistentes en las etapas posteriores del pipeline.

In [6]:
# =========================
# Diccionario de renombrado
# =========================
rename_dict = {
    "tourney_id": "identificador_torneo",
    "tourney_name": "nombre_torneo",
    "surface": "superficie",
    "draw_size": "tamano_cuadro",
    "tourney_level": "nivel_torneo",
    "tourney_date": "fecha_torneo",
    "match_num": "numero_partido",

    "winner_id": "identificador_ganador",
    "winner_seed": "cabeza_serie_ganador",
    "winner_entry": "tipo_entrada_ganador",
    "winner_name": "nombre_ganador",
    "winner_hand": "mano_dominante_ganador",
    "winner_ht": "altura_ganador",
    "winner_ioc": "pais_ganador",
    "winner_age": "edad_ganador",

    "loser_id": "identificador_perdedor",
    "loser_seed": "cabeza_serie_perdedor",
    "loser_entry": "tipo_entrada_perdedor",
    "loser_name": "nombre_perdedor",
    "loser_hand": "mano_dominante_perdedor",
    "loser_ht": "altura_perdedor",
    "loser_ioc": "pais_perdedor",
    "loser_age": "edad_perdedor",

    "score": "marcador_partido",
    "best_of": "numero_maximo_sets",
    "round": "ronda",
    "minutes": "duracion_partido_minutos",

    "w_ace": "aces_ganador",
    "w_df": "dobles_faltas_ganador",
    "w_svpt": "puntos_servicio_jugados_ganador",
    "w_1stIn": "primeros_saques_dentro_ganador",
    "w_1stWon": "puntos_ganados_primer_saque_ganador",
    "w_2ndWon": "puntos_ganados_segundo_saque_ganador",
    "w_SvGms": "juegos_servicio_ganador",
    "w_bpSaved": "break_points_salvados_ganador",
    "w_bpFaced": "break_points_enfrentados_ganador",

    "l_ace": "aces_perdedor",
    "l_df": "dobles_faltas_perdedor",
    "l_svpt": "puntos_servicio_jugados_perdedor",
    "l_1stIn": "primeros_saques_dentro_perdedor",
    "l_1stWon": "puntos_ganados_primer_saque_perdedor",
    "l_2ndWon": "puntos_ganados_segundo_saque_perdedor",
    "l_SvGms": "juegos_servicio_perdedor",
    "l_bpSaved": "break_points_salvados_perdedor",
    "l_bpFaced": "break_points_enfrentados_perdedor",

    "winner_rank": "ranking_ganador",
    "winner_rank_points": "puntos_ranking_ganador",
    "loser_rank": "ranking_perdedor",
    "loser_rank_points": "puntos_ranking_perdedor"
}

# =========================
#  Renombrar columnas
# =========================
matches_all = matches_all.rename(columns=rename_dict)

print("Dimensión tras renombrar:", matches_all.shape)
print("\nPrimeras columnas renombradas:")
print(matches_all.columns.tolist()[:20])



Dimensión tras renombrar: (47317, 54)

Primeras columnas renombradas:
['identificador_torneo', 'nombre_torneo', 'superficie', 'tamano_cuadro', 'nivel_torneo', 'fecha_torneo', 'numero_partido', 'identificador_ganador', 'cabeza_serie_ganador', 'tipo_entrada_ganador', 'nombre_ganador', 'mano_dominante_ganador', 'altura_ganador', 'pais_ganador', 'edad_ganador', 'identificador_perdedor', 'cabeza_serie_perdedor', 'tipo_entrada_perdedor', 'nombre_perdedor', 'mano_dominante_perdedor']


In [7]:
matches_all.head()

,identificador_torneo,nombre_torneo,superficie,tamano_cuadro,nivel_torneo,fecha_torneo,numero_partido,identificador_ganador,cabeza_serie_ganador,tipo_entrada_ganador,...,break_points_enfrentados_perdedor,ranking_ganador,puntos_ranking_ganador,ranking_perdedor,puntos_ranking_perdedor,match_year,es_muestra_estudio,es_buffer_carga,es_desarrollo_modelo,es_validacion_temporal
0,2008-1536,Madrid Masters,Hard,48,M,20081012,1,105208,NaN,NaN,...,5.0,54.0,719.0,27.0,1120.0,2008,0,1,0,0
1,2008-1536,Madrid Masters,Hard,48,M,20081012,2,103888,NaN,NaN,...,4.0,25.0,1145.0,59.0,694.0,2008,0,1,0,0
2,2008-1536,Madrid Masters,Hard,48,M,20081012,3,104259,NaN,NaN,...,9.0,31.0,1065.0,46.0,805.0,2008,0,1,0,0
3,2008-1536,Madrid Masters,Hard,48,M,20081012,4,103852,NaN,NaN,...,5.0,40.0,865.0,48.0,777.0,2008,0,1,0,0
4,2008-1536,Madrid Masters,Hard,48,M,20081012,5,103812,NaN,Q,...,8.0,73.0,579.0,26.0,1123.0,2008,0,1,0,0


In [8]:
matches_all.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47317 entries, 0 to 47316
Data columns (total 54 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   identificador_torneo                   47317 non-null  object 
 1   nombre_torneo                          47317 non-null  object 
 2   superficie                             47264 non-null  object 
 3   tamano_cuadro                          47317 non-null  int64  
 4   nivel_torneo                           47317 non-null  object 
 5   fecha_torneo                           47317 non-null  int64  
 6   numero_partido                         47317 non-null  int64  
 7   identificador_ganador                  47317 non-null  int64  
 8   cabeza_serie_ganador                   20264 non-null  float64
 9   tipo_entrada_ganador                   6166 non-null   object 
 10  nombre_ganador                         47317 non-null  object 
 11  ma

## 5. Exportación del conjunto consolidado

El conjunto resultante se exporta como archivo intermedio para su transformación posterior a la estructura jugador–partido.

In [9]:
csv_output = INTERIM_DIR / "atp_matches_2008_2019_2021_2024_full.csv"

matches_all.to_csv(
    csv_output,
    index=False,
    encoding="utf-8-sig"
)

print("Archivo CSV guardado en:")
print(csv_output.resolve())

Archivo CSV guardado en:
C:\Users\Usuario\Desktop\TFG\Datos\tennis_atp-master\interim\atp_matches_2008_2019_2021_2024_full.csv


## 6. Resultado de la etapa

Se obtiene un conjunto consolidado de 47.317 partidos y 54 variables, que integra las temporadas necesarias para el análisis y para la reconstrucción posterior del historial competitivo. El archivo `atp_matches_2008_2019_2021_2024_full.csv` constituye la entrada de la siguiente etapa, dedicada a la transformación de los registros originales a la estructura jugador–partido y a la preparación de los predictores tradicionales.